# Kapitel 4 — Mit Wahrscheinlichkeiten argumentieren · Kommentierte Lösung

*Mathe für KI, Schritt für Schritt*, Belk Siri — Begleit-Notebook.

[In Google Colab öffnen](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/de/ch04_corrige.ipynb) · Begleitseite: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/de.html

Jede Antwort beginnt mit der Überlegung auf Papier und zeigt dann ihre Entsprechung in Python. Bei jeder Berechnung nennen die Kommentare die Gruppe, die als Nenner dient.

**Inhalt.** Daten · Die Berechnungen der Lektionen 4.1 bis 4.5 · Aufgaben 4.1 bis 4.6

**Fragen Sie sich vor jeder Berechnung**, in welcher Menge Sie zählen: Spams, markierte Nachrichten oder empfangene Nachrichten.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Farben des Buchs: eine Farbe = eine Rolle
SPAM, LEGIT, FLAGGED, DENOM = '#d62728', '#b9b6ae', '#1f4e8c', '#ffd54f'
BLUE, ORANGE = '#2a78d6', '#eb6834'   # „kein Spam verpasst“, Kurve der Normalverteilung
from math import comb

## Daten

Die Kennungen D4.1 und D4.2 sind die des Buchs und der CSV-Dateien.

In [ ]:
# Daten von Kapitel 4, aus den CSV-Dateien des Buchs übernommen (nicht ändern)
# Zeilen: Spam, legitim. Spalten: markiert, nicht markiert.
ROWS, COLUMNS = ['spam', 'legitim'], ['markiert', 'nicht markiert']

# D4.1 — Postfach, in dem 20 % der Nachrichten Spam sind (1 000 Nachrichten)
inbox_20 = pd.DataFrame([[180, 20],
     [40, 760]],
                        index=ROWS, columns=COLUMNS)

# D4.2 — gleicher Filter, Postfach, in dem 2 % der Nachrichten Spam sind (1 000 Nachrichten)
inbox_2 = pd.DataFrame([[18, 2],
     [49, 931]],
                       index=ROWS, columns=COLUMNS)

# D4.3 — 1 000 Bearbeitungszeiten (ms), identisch zur CSV-Datei neu erzeugt
durations = np.random.default_rng(2026).normal(40, 5, 1000).round(1)

### Die Variablen der Daten

Die Namen im Code sind englisch. Für die Spalten gibt die letzte Spalte den Namen an, der in den CSV-Dateien des Buchs verwendet wird.

| Name | Bedeutung | Einheit | Spalte der CSV-Datei |
|:--|:--|:--|:--|
| `inbox_20` | D4.1: Postfach, in dem 20 % der Nachrichten Spam sind (Zeilen Spam / legitim, Spalten markiert / nicht markiert) | Nachrichten | — |
| `inbox_2` | D4.2: derselbe Filter, Postfach, in dem 2 % der Nachrichten Spam sind | Nachrichten | — |
| `durations` | D4.3: 1000 Bearbeitungszeiten | ms | — |

## Die Berechnungen der Lektionen in Python

Diese Zellen wiederholen die Berechnungen der Lektionen 4.1 bis 4.5.

In [ ]:
# Lektion 4.1 — zählen, dann durch die richtige Gruppe teilen (Gleichung 4.1)
t = inbox_20
n = t.values.sum()
print('gesamt:', n, '| Spams:', t.loc['spam'].sum(), '| markiert:', t['markiert'].sum())
print('P(Spam) =', t.loc['spam'].sum() / n, '| P(markiert) =', t['markiert'].sum() / n,
      '| P(Spam und markiert) =', t.loc['spam', 'markiert'] / n)
# Unter den markierten Nachrichten: Der Nenner ist die Spalte „markiert“
print(f"Spams unter den markierten: {t.loc['spam', 'markiert']} / {t['markiert'].sum()} "
      f"≈ {t.loc['spam', 'markiert'] / t['markiert'].sum():.2f}")

In [ ]:
# Lektion 4.2 — derselbe Filter in einem Postfach mit seltenem Spam (Gleichung 4.2)
d, f = 0.90, 0.05                   # markierte Anteile: der Spams, der legitimen Nachrichten
for p, t in [(0.20, inbox_20), (0.02, inbox_2)]:
    n_spam, n_leg = round(1000 * p), round(1000 * (1 - p))
    rebuilt = [[round(d * n_spam), n_spam - round(d * n_spam)], [round(f * n_leg), n_leg - round(f * n_leg)]]
    assert (np.array(rebuilt) == t.values).all()   # die Tafel ergibt sich aus den Raten
    print(f'Basisrate {p:.0%}: {t.values.tolist()}  →  Anteil der Spams unter den markierten '
          f'= {d * p / (d * p + f * (1 - p)):.4f}')

In [ ]:
# Lektion 4.3 — zwei bedingte Wahrscheinlichkeiten, zwei Nenner (Gleichung 4.3)
def conditional(t, case, group):
    """Anteil des Felds (Zeile, Spalte) in der Gruppe: eine Zeile ('spam') oder eine Spalte ('markiert')."""
    denominator = t.loc[group].sum() if group in t.index else t[group].sum()
    return t.loc[case] / denominator, denominator

for name, t in [('20 %', inbox_20), ('2 %', inbox_2)]:
    recall, den_r = conditional(t, ('spam', 'markiert'), 'spam')
    precision, den_p = conditional(t, ('spam', 'markiert'), 'markiert')
    print(f'Postfach mit {name}: P(markiert | Spam) = {recall:.2f} (Nenner {den_r}); '
          f'P(Spam | markiert) = {precision:.2f} (Nenner {den_p})')

# Dieselben Zahlen mit scikit-learn, aus Labels Nachricht für Nachricht (falls installiert)
try:
    from sklearn.metrics import precision_score, recall_score
    truth = [1] * 180 + [1] * 20 + [0] * 40 + [0] * 760      # 1 = Spam
    predicted_label = [1] * 180 + [0] * 20 + [1] * 40 + [0] * 760    # 1 = markiert
    print(f'scikit-learn: Recall = {recall_score(truth, predicted_label):.2f}, Präzision = {precision_score(truth, predicted_label):.2f}')
except ImportError:
    print('scikit-learn nicht installiert: Diese Berechnung ist optional.')

In [ ]:
# Lektion 4.4 — der Satz von Bayes (Gleichungen 4.4 und 4.5) und die Kurve des Buchs
def bayes(p, d=0.90, f=0.05):
    p_flagged = d * p + f * (1 - p)                  # Gleichung 4.5
    return d * p / p_flagged                        # Gleichung 4.4

for p in (0.20, 0.02):
    print(f'P(Spam) = {p}: P(markiert) = {0.9 * p + 0.05 * (1 - p):.3f}, P(Spam | markiert) = {bayes(p):.4f}')
P = np.linspace(0.001, 0.5, 500)
plt.figure(figsize=(6, 3))
plt.plot(100 * P, bayes(P), color=FLAGGED, linewidth=2)
plt.scatter([2, 20], [bayes(0.02), bayes(0.2)], marker='s', color=SPAM, zorder=3)
plt.xlabel('Basisrate: Anteil der Spams (%)'); plt.ylabel('P(Spam | markiert)'); plt.grid(alpha=0.4)
plt.show()

In [ ]:
# Lektion 4.5 — zehn Spams: Binomialverteilung (Gleichung 4.7) und Gegenereignis (Gleichung 4.8)
n, p = 10, 0.1                        # wir zählen über 10 SPAM-Nachrichten; p = Wahrscheinlichkeit, verpasst zu werden
distribution = [comb(n, k) * p ** k * (1 - p) ** (n - k) for k in range(n + 1)]
print('0,9^10 =', round(0.9 ** 10, 4), '| mindestens einer verpasst =', round(1 - 0.9 ** 10, 4))
print('P(X = k), k = 0 bis 4:', [round(v, 3) for v in distribution[:5]], '| Summe =', round(sum(distribution), 6))
colors = [BLUE] + [SPAM] * n
plt.figure(figsize=(6, 3))
plt.bar(range(n + 1), distribution, color=colors, hatch=[''] + ['//'] * n, edgecolor='white')
plt.xlabel('Anzahl verpasster Spams unter 10 Spams'); plt.ylabel('Wahrscheinlichkeit'); plt.show()

# Ein Blick auf die Normalverteilung: Bearbeitungszeiten (D4.3)
m, s = durations.mean(), durations.std()
print(f'Mittelwert = {m:.2f} ms, Standardabweichung = {s:.2f} ms; innerhalb ±1 Standardabweichung: {np.mean(abs(durations - m) <= s):.1%}, '
      f'innerhalb ±2 Standardabweichungen: {np.mean(abs(durations - m) <= 2 * s):.1%}')

## Aufgabe 4.1 — Ein Diagramm lesen: zwanzig Spams

*Begriffe: Binomialverteilung, Gegenereignis (Lektion 4.5).* Lesen Sie zuerst die Abbildung im Buch.

### a) bis d) Kommentierte Lösung

Hier zählen wir über **20 Spams**, nicht über empfangene Nachrichten.

**a)** Kein Spam verpasst: etwa 0,36. Genau einer: etwa 0,38.

**b)** Der höchste Balken ist der bei 1: Ein verpasster Spam ist der wahrscheinlichste Fall.

**c)** $1 - 0{,}36 \approx 0{,}64$. Mit Gleichung 4.8: $1 - 0{,}95^{20} \approx 0{,}6415$.

**d)** Jeder Spam kommt nur halb so oft durch (5 % statt 10 %), aber es gibt doppelt so viele Spams (20 statt 10). Im Mittel verpasst der Filter immer noch einen: $20 \times 0{,}05 = 1$. Die Wahrscheinlichkeit, dass mindestens einer durchkommt, bleibt daher ähnlich: 64 % gegenüber 65 %.

In [ ]:
def binomial(n, p, k):
    return comb(n, k) * p ** k * (1 - p) ** (n - k)

In [ ]:
# Prüfung an den 20 Spams der Aufgabe
if binomial(20, 0.05, 0) is None:
    print('Ergänzen Sie die Funktion.')
else:
    p0, p1 = binomial(20, 0.05, 0), binomial(20, 0.05, 1)
    print(f'P(X = 0) = {p0:.4f}, P(X = 1) = {p1:.4f}, mindestens einer = {1 - p0:.4f} →',
          'richtig' if abs(p0 - 0.95 ** 20) < 1e-9 and abs(p1 - 20 * 0.05 * 0.95 ** 19) < 1e-9 else 'bitte prüfen')

## Aufgabe 4.2 — Ein vorsichtigerer Filter

*Begriffe: bedingte Wahrscheinlichkeit (Lektion 4.3), Satz von Bayes (Lektion 4.4), Basisrate (Lektion 4.2).* Der vorsichtige Filter markiert 75 % der Spams und 1 % der legitimen Nachrichten. Postfach mit 10 000 Nachrichten, davon 10 % Spam. Lösen Sie zuerst die Fragen a) bis e) auf Papier.

### a) bis c) Kommentierte Lösung

**a)** Beginnen Sie mit den Zeilen, also damit, was die Nachrichten sind. 10 % von 10 000 ergeben **1 000 Spams**, es bleiben **9 000 legitime Nachrichten**. Der Filter markiert 75 % der Spams: 0,75 × 1 000 = **750**, also kommen 250 Spams durch. Er markiert 1 % der legitimen Nachrichten: 0,01 × 9 000 = **90** Fehlalarme, und 8 910 legitime Nachrichten kommen durch.

| | Markiert | Nicht markiert | Gesamt |
|:--|--:|--:|--:|
| Spam | 750 | 250 | 1 000 |
| Legitim | 90 | 8 910 | 9 000 |
| **Gesamt** | **840** | **9 160** | **10 000** |

**b)** $P(\text{markiert} \mid \text{spam})$: Wir zählen **in der Spam-Zeile**, die als Nenner dient. $750 / 1\,000 = 0{,}75$. Wir erhalten die angegebene Rate zurück, das ist beruhigend. $P(\text{spam} \mid \text{markiert})$: Wir zählen **in der Spalte der markierten Nachrichten**. $750 / 840 \approx 0{,}893$. Gleicher Zähler, 750; verschiedene Nenner, 1 000 und 840.

**c)** Mit Gleichung 4.5: $P(\text{markiert}) = 0{,}75 \times 0{,}1 + 0{,}01 \times 0{,}9 = 0{,}075 + 0{,}009 = 0{,}084$, also die 840 markierten Nachrichten von 10 000. Mit Gleichung 4.4: $P(\text{spam} \mid \text{markiert}) = 0{,}075 / 0{,}084 \approx 0{,}893$. Bayes schreibt nur das Zählen in der Tafel mit Anteilen um.

### a) Die Tafel in Python

In [ ]:
spam_flagged = round(0.75 * 1000)          # 750: 75 % der 1 000 Spams
spam_not_flagged = 1000 - spam_flagged     # 250
legit_flagged = round(0.01 * 9000)      # 90 Fehlalarme: 1 % der 9 000 legitimen Nachrichten
legit_not_flagged = 9000 - legit_flagged   # 8 910

In [ ]:
# Prüfung
cells = [spam_flagged, spam_not_flagged, legit_flagged, legit_not_flagged]
if None in cells:
    print('Ergänzen Sie die vorige Zelle.')
else:
    cautious = pd.DataFrame([cells[:2], cells[2:]], index=ROWS, columns=COLUMNS)
    print(cautious)
    print('Tafel:', 'richtig' if cells == [750, 250, 90, 8910] else 'bitte prüfen')

### b) Die beiden bedingten Wahrscheinlichkeiten

In [ ]:
p_flagged_if_spam = 750 / (750 + 250)   # Nenner: die Spam-Zeile
p_spam_if_flagged = 750 / (750 + 90)    # Nenner: die Spalte der markierten

In [ ]:
# Prüfung
for name, yours, truth in [('P(markiert | Spam)', p_flagged_if_spam, 0.75),
                            ('P(Spam | markiert)', p_spam_if_flagged, 750 / 840)]:
    print(f'{name}:', 'noch nicht ausgefüllt' if yours is None else
          ('richtig' if abs(yours - truth) < 0.001 else 'bitte prüfen'))

### c) bis e) Der Satz von Bayes als Funktion

In [ ]:
def flagged_spam_probability(p, d, f):
    p_flagged = d * p + f * (1 - p)     # Gleichung 4.5: die beiden Felder der Spalte „markiert“
    return d * p / p_flagged            # Gleichung 4.4: das rote Feld geteilt durch die Spalte

In [ ]:
# Prüfung von c), d) und e)
if flagged_spam_probability(0.1, 0.75, 0.01) is None:
    print('Ergänzen Sie die Funktion.')
else:
    scenario = [('c) vorsichtig, 10 % Spam', (0.10, 0.75, 0.01), 0.8929),
           ('d) vorsichtig, 1 % Spam', (0.01, 0.75, 0.01), 0.4310),
           ('e) vorsichtig, 2 % Spam', (0.02, 0.75, 0.01), 0.6048),
           ('e) Filter der Lektionen, 2 % Spam', (0.02, 0.90, 0.05), 0.2687)]
    for name, args, expected in scenario:
        v = flagged_spam_probability(*args)
        print(f'{name}: {v:.4f} →', 'richtig' if abs(v - expected) < 0.0001 else 'bitte prüfen')
    P = np.linspace(0.001, 0.5, 500)
    plt.figure(figsize=(6, 3))
    plt.plot(100 * P, flagged_spam_probability(P, 0.90, 0.05), color=FLAGGED, linewidth=2,
             label='Filter der Lektionen (90 %; 5 %)')
    plt.plot(100 * P, flagged_spam_probability(P, 0.75, 0.01), color=SPAM, linewidth=2, linestyle='--',
             label='vorsichtiger Filter (75 %; 1 %)')
    plt.axvline(2, color='black', linewidth=0.8, linestyle=':')
    plt.xlabel('Basisrate: Anteil der Spams (%)'); plt.ylabel('P(Spam | markiert)')
    plt.legend(); plt.grid(alpha=0.4); plt.show()

### d) und e) Kommentierte Lösung

**d)** Erwartete Vorhersage: Die Wahrscheinlichkeit **sinkt**, da Spams seltener werden und Fehlalarme stärker ins Gewicht fallen (Lektion 4.2). Für 10 000 Nachrichten: 100 Spams, davon 75 markiert; 9 900 legitime Nachrichten, davon 99 markiert. In der Spalte der markierten: $75 / (75 + 99) = 75 / 174 \approx 0{,}431$. Weniger als jede zweite Markierung ist echter Spam, obwohl sich der Filter nicht verändert hat.

**e)** Für den vorsichtigen Filter im Postfach mit 2 %: $\dfrac{0{,}75 \times 0{,}02}{0{,}75 \times 0{,}02 + 0{,}01 \times 0{,}98} = \dfrac{0{,}015}{0{,}0248} \approx 0{,}605$, gegenüber 0,27 für den Filter der Lektionen. Diese beiden Zahlen zu vergleichen heißt, die **Präzision** der beiden Filter zu vergleichen: Wir zählen in der Spalte der markierten Nachrichten. Wenn der vorsichtige Filter eine Nachricht markiert, können Sie sich eher darauf verlassen: Er macht fünfmal weniger Fehlalarme. Aber um einen Filter zu wählen, reicht die Präzision nicht. Zählen Sie auch in der Spam-Zeile: Von 100 Spams verpasst der vorsichtige Filter 25, der Filter der Lektionen 10 (sein **Recall** beträgt 0,90 gegenüber 0,75). Die richtige Wahl hängt davon ab, was teurer ist: eine wichtige Nachricht, die im „Junk“-Ordner verloren geht, oder ein Spam mehr im Posteingang. Wie in Kapitel 3 muss man das Maß festlegen, bevor man vergleicht. Die Kurve zeigt, dass der Abstand zwischen den beiden Filtern am größten ist, wenn Spams selten sind.

## Aufgabe 4.3 — Drei Spams, dann zehn Nachrichten

*Begriffe: Bernoulli-Experiment und Binomialverteilung (Lektion 4.5), bedingte Wahrscheinlichkeit (Lektion 4.3).* Lösen Sie die Fragen zuerst auf Papier.

### a) bis e) Kommentierte Lösung

**a)** Jeder Zweig multipliziert 0,9 (markiert) oder 0,1 (verpasst). Der Ausgang „drei markiert“ hat $0{,}9^3 = 0{,}729$; jeder Ausgang mit genau einem verpassten Spam hat $0{,}1 \times 0{,}9^2 = 0{,}081$; mit zwei verpassten $0{,}1^2 \times 0{,}9 = 0{,}009$; alle drei verpasst $0{,}1^3 = 0{,}001$.

**b)** Drei Ausgänge haben genau einen verpassten Spam: $P(X = 1) = 3 \times 0{,}081 = 0{,}243$. Ebenso $P(X = 2) = 3 \times 0{,}009 = 0{,}027$, $P(X = 0) = 0{,}729$, $P(X = 3) = 0{,}001$. Summe: $0{,}729 + 0{,}243 + 0{,}027 + 0{,}001 = 1$. Der Faktor 3 ist der Koeffizient $\binom{3}{1} = \binom{3}{2} = 3$.

**c)** Durch Addieren: $0{,}243 + 0{,}027 + 0{,}001 = 0{,}271$. Über das Gegenereignis: $1 - 0{,}729 = 0{,}271$. Die zweite Rechnung ist kürzer.

**d)** **Prüfen Sie, in welcher Menge Sie zählen.** Wir zählen nicht mehr über zehn *Spams*, sondern über zehn *empfangene Nachrichten*, von denen einige legitim sind. Für eine empfangene Nachricht bedeutet „Spam, der durchkommt“ „Spam **und** nicht markiert“: $0{,}2 \times 0{,}1 = 0{,}02$ (Lektion 4.3). Keine der zehn kommt mit der Wahrscheinlichkeit $0{,}98^{10} \approx 0{,}817$ durch, also kommt mindestens ein Spam mit der Wahrscheinlichkeit $1 - 0{,}98^{10} \approx 0{,}183$ durch.

**e)** In Lektion 4.5 sind alle zehn Nachrichten Spams: Jede kommt mit 10 % Wahrscheinlichkeit durch. Unter zehn empfangenen Nachrichten sind im Mittel nur zwei Spams, und jede Nachricht ist nur mit 2 % Wahrscheinlichkeit ein Spam, der durchkommt. „Zehn Spams“ und „zehn empfangene Nachrichten“ sind zwei verschiedene Fragen.

In [ ]:
distribution_3 = [comb(3, k) * 0.1 ** k * 0.9 ** (3 - k) for k in range(4)]   # über 3 SPAM-Nachrichten
p_d = 1 - (1 - 0.2 * 0.1) ** 10   # über 10 EMPFANGENE NACHRICHTEN: Spam UND nicht markiert = 0,02

In [ ]:
# Prüfung
if distribution_3 is None or p_d is None:
    print('Ergänzen Sie die vorige Zelle.')
else:
    print('b):', [round(v, 3) for v in distribution_3], '→', 'richtig' if np.allclose(distribution_3, [0.729, 0.243, 0.027, 0.001]) else 'bitte prüfen')
    print(f'd): {p_d:.4f} →', 'richtig' if abs(p_d - 0.1829) < 0.0001 else 'bitte prüfen')

## Aufgabe 4.4 — Zehn Spams simulieren

*Begriffe: Binomialverteilung (Lektion 4.5), Häufigkeit und Wahrscheinlichkeit.* Wir zählen über **zehn Spams**.

In [ ]:
def series_simulation(rng):
    """Gibt die Anzahl verpasster Spams unter 10 zurück. rng.random() zieht eine Zahl zwischen 0 und 1."""
    return int(np.sum(rng.random(10) < 0.1))   # eine Ziehung pro Spam: verpasst, wenn < 0,1

In [ ]:
# b) und c) 10 000 Serien (fester Seed, um dieselben Zahlen zu erhalten)
rng = np.random.default_rng(4)
if series_simulation(rng) is None:
    print('Ergänzen Sie die Funktion.')
else:
    rng = np.random.default_rng(4)
    X = np.array([series_simulation(rng) for _ in range(10000)])
    freq = np.mean(X >= 1)
    print(f'beobachtete Häufigkeit: {freq:.4f}; berechnete Wahrscheinlichkeit: {1 - 0.9 ** 10:.4f} →',
          'richtig' if abs(freq - (1 - 0.9 ** 10)) < 0.02 else 'bitte prüfen')
    k = np.arange(6)
    plt.figure(figsize=(6, 3))
    plt.bar(k - 0.2, [np.mean(X == v) for v in k], width=0.4, color=FLAGGED, label='beobachtete Häufigkeit')
    plt.bar(k + 0.2, [comb(10, v) * 0.1 ** v * 0.9 ** (10 - v) for v in k], width=0.4, color=SPAM,
            hatch='//', edgecolor='white', label='Wahrscheinlichkeit (Gleichung 4.7)')
    plt.xlabel('Verpasste Spams unter 10'); plt.legend(); plt.show()
    # d) Mehr Serien: Die Häufigkeit nähert sich der Wahrscheinlichkeit
    for N in (100, 100, 100, 100000):
        Y = np.array([series_simulation(rng) for _ in range(N)])
        print(f'{N:>6} Serien: Häufigkeit = {np.mean(Y >= 1):.4f}')

### Kommentierte Lösung

Mit dem Seed des Notebooks ergeben 10 000 Serien eine Häufigkeit von 0,6549, bei einer Wahrscheinlichkeit von 0,6513. Beide liegen nahe beieinander, sind aber nicht gleich: Eine beobachtete Häufigkeit schwankt von Simulation zu Simulation. Bei 100 Serien kann sie um mehrere Prozentpunkte abweichen (führen Sie die Zelle ohne Seed erneut aus, um das zu sehen). Bei 100 000 Serien kommt sie bis auf wenige Tausendstel heran. Die Wahrscheinlichkeit ist der Wert, gegen den die Häufigkeit strebt, wenn die Anzahl der Serien wächst. Prüfen Sie auch die Menge: Jede Serie zählt **Spams**, nie empfangene Nachrichten.

## Aufgabe 4.5 — Vorhersagen, dann prüfen: eine Spam-Kampagne

*Begriffe: Unabhängigkeit, Gegenereignis (Lektion 4.5).*

### a) bis c) und e) Kommentierte Lösung

**b)** Der Filter verpasst alle zehn Spams auf einmal mit Wahrscheinlichkeit 0,1, oder er verpasst keinen. Die Anzahl verpasster Spams ist also 0 oder 10. Mindestens einer kommt mit Wahrscheinlichkeit **0,1** durch, viel weniger als 65 %.

**c)** Im Mittel: $10 \times 0{,}1 + 0 \times 0{,}9 = 1$ verpasster Spam. Das ist derselbe Mittelwert wie in Lektion 4.5 ($10 \times 0{,}1 = 1$). Der Mittelwert ist gleich, aber die Verteilung ist ganz anders.

**e)** Die Unabhängigkeit gilt nicht mehr: Zu wissen, dass der erste Spam verpasst wird, macht es fast sicher, dass auch die anderen verpasst werden. Die Wahrscheinlichkeit, verpasst zu werden, 0,1 für jeden, hat sich nicht verändert. Für die Nutzerin oder den Nutzer sind zehn Spams auf einmal lästiger und sichtbarer. Die Binomialverteilung gilt nur, wenn ihre beiden Annahmen erfüllt sind.

In [ ]:
def campaign_simulation(rng):
    return 10 if rng.random() < 0.1 else 0   # eine einzige Ziehung für die ganze Kampagne

In [ ]:
# d) Prüfung durch Simulation
rng = np.random.default_rng(5)
if campaign_simulation(rng) is None:
    print('Ergänzen Sie die Funktion.')
else:
    rng = np.random.default_rng(5)
    Z = np.array([campaign_simulation(rng) for _ in range(10000)])
    print(f'mindestens einer verpasst: {np.mean(Z >= 1):.4f}; Mittelwert: {Z.mean():.3f} →',
          'richtig' if abs(np.mean(Z >= 1) - 0.1) < 0.01 and abs(Z.mean() - 1) < 0.1 else 'bitte prüfen')

## Aufgabe 4.6 — Herausforderung: Ab wie vielen Spams liegt das Risiko, einen zu verpassen, über 99 %?

*Begriffe: Bernoulli- und Binomialverteilung, Normalverteilung (Lektion 4.5), Standardabweichung (Lektion 1.1).*

### a) bis d) Kommentierte Lösung

**a)** Sei $X_i = 1$, wenn Spam $i$ verpasst wird, sonst 0. Dann gilt $X = X_1 + \dots + X_n$: Das Addieren der Einsen zählt die verpassten Spams. Jedes $X_i$ hat den Mittelwert $1 \times p + 0 \times (1 - p) = p$, und der Mittelwert einer Summe ist die Summe der Mittelwerte: $n\,p$. Für 10 Spams also 1.

**b)** Es muss $1 - 0{,}9^n > 0{,}99$ gelten, also $0{,}9^n < 0{,}01$. Man findet $0{,}9^{43} \approx 0{,}0108$ und $0{,}9^{44} \approx 0{,}0097$: Es braucht **44 Spams**.

**c)** Mittelwert: $100 \times 0{,}1 = 10$. Standardabweichung: $\sqrt{100 \times 0{,}1 \times 0{,}9} = \sqrt{9} = 3$.

**d)** Die Balken der Binomialverteilung schmiegen sich fast an die Glockenkurve mit Mittelwert 10 und Standardabweichung 3. Wenn $n$ groß ist, ähnelt die Binomialverteilung einer Normalverteilung: Deshalb wird die Kurve aus Lektion 4.5 oft wiederkehren.

In [ ]:
n_min = next(n for n in range(1, 200) if 0.9 ** n < 0.01)   # wir probieren n = 1, 2, 3…
mean_100 = 100 * 0.1
std_100 = np.sqrt(100 * 0.1 * 0.9)

In [ ]:
# Prüfung, dann d) Binomialverteilung gegen Normalverteilung
if None in (n_min, mean_100, std_100):
    print('Ergänzen Sie die vorige Zelle.')
else:
    print(f'n_min = {n_min} →', 'richtig' if n_min == 44 else 'bitte prüfen')
    print(f'Mittelwert = {mean_100:.1f}, Standardabweichung = {std_100:.1f} →',
          'richtig' if np.isclose(mean_100, 10) and np.isclose(std_100, 3) else 'bitte prüfen')
    k = np.arange(0, 26)
    plt.figure(figsize=(6, 3))
    plt.bar(k, [comb(100, v) * 0.1 ** v * 0.9 ** (100 - v) for v in k], color=SPAM, alpha=0.8,
            label='Binomialverteilung (n = 100, p = 0,1)')
    xs = np.linspace(0, 25, 300)
    plt.plot(xs, np.exp(-(xs - 10) ** 2 / 18) / (3 * np.sqrt(2 * np.pi)), color=ORANGE, linewidth=2,
             label='Normalverteilung (Mittelwert 10, Standardabweichung 3)')
    plt.xlabel('Verpasste Spams unter 100'); plt.legend(); plt.show()